In [1]:
import torch
import numpy as np
import cvxpy as cp

In [2]:
import os
print(os.getenv('MOSEKLM_LICENSE_FILE'))
def torch_to_np(x):
    return x.detach().numpy()

/workspaces/cvxpylayers/.devcontainer/attach/mosek.lic


In [3]:

# 定义优化问题（示例）
n, m = 5, 3
x = cp.Variable(n)
A = cp.Parameter((m, n))
b = cp.Parameter(m)
constraints = [A @ x <= b, x >= 0]
objective = cp.Minimize(cp.sum_squares(x[:2]) + 0.1 * cp.norm1(x[2:]))
problem = cp.Problem(objective, constraints)


这个优化问题，不同求解器和CVXPYLAYER 表现如何？CVXPYLAYER最终调用了哪个求解器？

In [4]:
from cvxpylayers.torch  import CvxpyLayer
A.value = np.random.randn(m, n)  # 生成随机矩阵
b.value = np.abs(np.random.randn(m))  # 生成非负随机向量
A.torch = torch.from_numpy(A.value)
b.torch = torch.from_numpy(b.value)
layer = CvxpyLayer(
    problem=problem,
    parameters=[A, b],
    variables=[x]
)
cp_mosek=problem.solve(solver=cp.MOSEK)
cp_SCS=problem.solve(solver=cp.SCS)
# 这里可能是最优点坐标,,由于 diffcp 返回的 xs 是最优点的坐标。
cp_cvxpylayer_default, =layer(A.torch, b.torch)
cp_cvxpylayer_default_np = torch_to_np(cp_cvxpylayer_default)
cp_cvxpy_objective_expr = objective.expr.value
A.value,A.torch,b.value,b.torch,problem,f"==>SCS:  {cp_SCS}",f"==>MOSEK:{cp_mosek}",cp_cvxpylayer_default,cp_cvxpylayer_default_np,f"==>CL:   {cp_cvxpy_objective_expr}"


(array([[-7.83055372e-04,  2.74545087e+00, -3.66326081e-01,
         -6.15813450e-01, -9.62804225e-02],
        [ 3.74061780e-01,  2.44948145e-01, -5.78530692e-01,
         -1.07949056e+00, -1.41335929e-01],
        [-9.35130934e-01, -2.11424661e-01, -9.72463721e-01,
         -8.13432608e-01, -4.98501430e-01]]),
 tensor([[-7.8306e-04,  2.7455e+00, -3.6633e-01, -6.1581e-01, -9.6280e-02],
         [ 3.7406e-01,  2.4495e-01, -5.7853e-01, -1.0795e+00, -1.4134e-01],
         [-9.3513e-01, -2.1142e-01, -9.7246e-01, -8.1343e-01, -4.9850e-01]],
        dtype=torch.float64),
 array([0.21483531, 0.21332099, 0.20237202]),
 tensor([0.2148, 0.2133, 0.2024], dtype=torch.float64),
 Problem(Minimize(Expression(CONVEX, NONNEGATIVE, ())), [Inequality(Expression(AFFINE, UNKNOWN, (3,))), Inequality(Constant(CONSTANT, ZERO, ()))]),
 '==>SCS:  8.495362831258339e-09',
 '==>MOSEK:7.4176990143743805e-09',
 tensor([1.4136e-05, 1.8748e-05, 2.9998e-07, 3.7910e-07, 1.1468e-07],
        dtype=torch.float64),
 array

### 或许，可以将问题的CONE 形式从 CVXPYLAYER 套出来

class CvxpyLayer(torch.nn.Module) -> def forward(self, *params, solver_args={}) ->_CvxpyLayerFn().__call__ = class _CvxpyLayerFnFn(torch.autograd.Function).appy -> def forward(ctx, *params):batch整形 后

进入_forward_numpy(params_numpy, context) 执行数值计算，这个是_CvxpyLayerFn()的初始化参数之一，来自class CvxpyLayer(torch.nn.Module):的初始化参数，默认是 utils.py 中的def forward_numpy(params_numpy, context):

整形的结果是形成了 context 和 params_numpy（params 的逐个numpy） ，优化问题的格式和求解要求 （ctx） 都在 context 中

In [76]:
import numpy as np
import diffcp
import time
from dataclasses import dataclass
from typing import Any

def forward_numpy(params_numpy, context):
    """Forward pass in numpy."""

    info = {}

    if context.gp:
        param_map = {}
        # construct a list of params for the DCP problem
        for param, value in zip(context.param_order, params_numpy):
            if param in context.old_params_to_new_params:
                new_id = context.old_params_to_new_params[param].id
                param_map[new_id] = np.log(value)
            else:
                new_id = param.id
                param_map[new_id] = value
        params_numpy = [param_map[pid] for pid in context.param_ids]

    # canonicalize problem
    start = time.time()
    As, bs, cs, cone_dicts, shapes = [], [], [], [], []
    for i in range(context.batch_size):
        params_numpy_i = [
            p if sz == 0 else p[i] for p, sz in zip(params_numpy, context.batch_sizes)
        ]
        c, _, neg_A, b = context.compiler.apply_parameters(
            dict(zip(context.param_ids, params_numpy_i)), keep_zeros=True
        )
        A = -neg_A  # cvxpy canonicalizes -A
        As.append(A)
        bs.append(b)
        cs.append(c)
        cone_dicts.append(context.cone_dims)
        shapes.append(A.shape)
    info["canon_time"] = time.time() - start
    info["shapes"] = shapes

    # compute solution and derivative function
    start = time.time()
    try:
        if context.solve_and_derivative:
            xs, _, _, _, DT_batch = diffcp.solve_and_derivative_batch(
                As, bs, cs, cone_dicts, **context.solver_args
            )
            info["DT_batch"] = DT_batch
        else:
            xs, _, _ = diffcp.solve_only_batch(
                As, bs, cs, cone_dicts, **context.solver_args
            )
    except diffcp.SolverError as e:
        print(
            "Please consider re-formulating your problem so that "
            "it is always solvable or increasing the number of "
            "solver iterations."
        )
        raise e
    info["solve_time"] = time.time() - start

    # extract solutions and append along batch dimension
    start = time.time()
    sol = [[] for i in range(len(context.variables))]
    for i in range(context.batch_size):
        sltn_dict = context.compiler.split_solution(xs[i], active_vars=context.var_dict)
        for j, v in enumerate(context.variables):
            sol[j].append(np.expand_dims(sltn_dict[v.id], axis=0))
    sol = [np.concatenate(s, axis=0) for s in sol]

    if not context.batch:
        sol = [np.squeeze(s, axis=0) for s in sol]

    if context.gp:
        sol = [np.exp(s) for s in sol]
        info["sol"] = sol

    return sol, info

In [ ]:
import mosek

try:
    env = mosek.Env()
    print("Mosek 证书已正确配置。")
except mosek.Error as e:
    print(f"Mosek 证书配置错误: {e}")

Mosek 证书已正确配置。


In [ ]:
from cvxpylayers.torch import CvxpyLayer

In [ ]:
### PREAMBLE
# Differentiable Convex Optimization Layers
# CVXPY creates powerful new PyTorch and TensorFlow layers
# Akshay Agrawal, Brandon Amos, Shane Barratt, Stephen Boyd, Steven Diamond, J. Zico Kolter
# wideimg: ./overview.png

import numpy as np
import numpy.random as npr

import torch
from torch import nn
import torch.nn.functional as F

import os
import sys
import shutil

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
plt.style.use('bmh')
from matplotlib import rc
# rc('font',**{'family':'sans-serif','sans-serif':['Helvetica']})
# rc('text', usetex=True)

def to_np(x):
    return x.detach().numpy()

def inConstraints(x, G, h):
    return int(np.all(G.dot(x) <= h))

def plotConstraints(G0, h0, G1=None, h1=None, xmin=0, xmax=1, ymin=0, ymax=1,
                    point=None,
                    pointy=None):
    xx, yy = np.meshgrid(np.linspace(xmin, xmax, 600),
                         np.linspace(ymin, ymax, 600))
    xxFlat = xx.ravel()
    yyFlat = yy.ravel()
    gridX = np.vstack((xxFlat, yyFlat)).T

    fig, ax = plt.subplots(1, 1, figsize=(5,5))
    ax.axis([xmin, xmax, ymin, ymax])

    zzFlat0 = []
    zzFlat1 = []
    _xmin = _xmax = _ymin = _ymax = 0.5
    for i in range(len(gridX)):
        xi = gridX[i]
        t = inConstraints(xi, G0, h0)
        zzFlat0.append(t)
        if t:
            _xmin = min(_xmin, xi[0])
            _xmax = max(_xmax, xi[0])
            _ymin = min(_ymin, xi[1])
            _ymax = max(_ymax, xi[1])
        if G1 is not None:
            t = inConstraints(xi, G1, h1)
            zzFlat1.append(t)

    zz0 = np.array(zzFlat0).reshape(xx.shape)
    cs = ax.contourf(xx, yy, zz0, cmap=cm.Blues, alpha=0.5)
    cs.cmap.set_under('white')
    cs.set_clim(0.5, 1.0)

    if G1 is not None:
        zz1 = np.array(zzFlat1).reshape(xx.shape)
        cs = ax.contourf(xx, yy, zz1, cmap=cm.Reds, alpha=0.5)
        cs.cmap.set_under('white')
        cs.set_clim(0.5, 1.0)

    scale = 0.1
    _xmin, _ymin = [z-scale*z for z in [_xmin, _ymin]]
    _xmax, _ymax = [z+scale*z for z in [_xmax, _ymax]]
    ax.axis([_xmin, _xmax, _ymin, _ymax])

    if point is not None:
        x = round(to_np(point)[0]*600)
        y = round(to_np(point)[1]*600)
        print(f"Point: {to_np(point)}, grid: {x},{y}")
        ax.scatter(x,y,c='red', s=80,zorder=3)
         # 扩展坐标轴范围确保包含红点
        ax.set_xlim(min(_xmin, x), max(_xmax, point[0]))
        ax.set_ylim(min(_ymin, y), max(_ymax, point[1]))
    if pointy is not None:
        x = round(to_np(pointy)[0]*600)
        y = round(to_np(pointy)[1]*600)
        print(f"Pointy: {to_np(pointy)}, grid: {x},{y}")
        ax.scatter(x,y,c='green', s=80,zorder=3)
         # 扩展坐标轴范围确保包含红点
        ax.set_xlim(min(_xmin, point[0]), max(_xmax, point[0]))
        ax.set_ylim(min(_ymin, point[1]), max(_ymax, point[1]))

    ax.axes.get_xaxis().set_visible(False)
    ax.axes.get_yaxis().set_visible(False)



    return fig, ax

%matplotlib inline
%config InlineBackend.figure_format = 'svg'

In [ ]:
nx, ncon = 2, 10

_G = cp.Parameter((ncon, nx))
_h = cp.Parameter(ncon)
_x = cp.Parameter(nx)
_y = cp.Variable(nx)
obj = cp.Minimize(0.5*cp.sum_squares(_x-_y))
cons = [_G @ _y <= _h]
prob = cp.Problem(obj, cons)

layer = CvxpyLayer(prob, parameters=[_G, _h, _x], variables=[_y])

In [ ]:
torch.manual_seed(6)
G = torch.FloatTensor(ncon, nx).uniform_(-4, 4)
G
z0 = torch.full([nx], 0.5)
s0 = torch.full([ncon], 0.5)
h = G.mv(z0)+s0
plotConstraints(to_np(G), to_np(h),point=torch.randn(nx).clamp(min=0, max=1))

torch.manual_seed(22)
G_hat = nn.Parameter(torch.FloatTensor(ncon, nx).uniform_(-4, 4).requires_grad_())
h_hat = G_hat.mv(z0)+s0
plotConstraints(to_np(G), to_np(h), to_np(G_hat), to_np(h_hat))

In [ ]:
x = torch.randn(nx).clamp(min=0, max=1)
y, = layer(G,h,x)
plotConstraints(to_np(G), to_np(h),point=x,pointy=y)